In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt

In [5]:
data = pd.read_csv('./weatherHistory.csv')

In [6]:
data['Formatted Date'] = pd.to_datetime(data['Formatted Date'], errors='coerce', utc=True)
data = data.dropna(subset=['Formatted Date'])
data['Formatted Date'] = data['Formatted Date'].dt.tz_convert(None)

# Seleccionar X y y
y = data['Temperature (C)'].values.reshape(-1, 1)
X = data.drop(columns=['Temperature (C)', 'Formatted Date'])

# Columnas categóricas
cat_cols = ['Summary', 'Precip Type', 'Daily Summary']
num_cols = [col for col in X.columns if col not in cat_cols]

# One-hot encode de categóricas
ohe = OneHotEncoder(sparse=False)
X_cat = ohe.fit_transform(X[cat_cols])

# Escalar numéricas
scaler = MinMaxScaler()
X_num = scaler.fit_transform(X[num_cols])

# Concatenar numéricas + categóricas
X_all = np.hstack([X_num, X_cat])

# Escalar y
y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(y)

TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_all, y_scaled, test_size=0.2, random_state=42)

# Convertir a tensores
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
input_size = X_train.shape[1]

class DenseNet(nn.Module):
    def __init__(self, input_size, hidden1=128, hidden2=64):
        super(DenseNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, 1)
        )
    def forward(self, x):
        return self.model(x)

model = DenseNet(input_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
EPOCHS = 100
for epoch in range(1, EPOCHS+1):
    model.train()
    optimizer.zero_grad()
    y_pred = model(X_train_t)
    loss = criterion(y_pred, y_train_t)
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_pred = model(X_test_t)
            val_loss = criterion(val_pred, y_test_t)
        # Calcular "accuracy" aproximado como % de predicción dentro de ±1°C
        y_pred_rescaled = y_scaler.inverse_transform(y_pred.detach().numpy())
        y_test_rescaled = y_scaler.inverse_transform(y_test_t.numpy())
        acc = np.mean(np.abs(y_pred_rescaled - y_test_rescaled) < 1.0) * 100
        val_acc = np.mean(np.abs(y_scaler.inverse_transform(val_pred.numpy()) - y_test_rescaled) < 1.0) * 100
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {loss.item():.5f} | Val Loss: {val_loss.item():.5f} | acc: {acc:.2f}% | val_acc: {val_acc:.2f}%")



In [ ]:
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_t).numpy()
y_test_pred_rescaled = y_scaler.inverse_transform(y_test_pred)
y_test_rescaled = y_scaler.inverse_transform(y_test_t.numpy())

plt.figure(figsize=(12,5))
plt.plot(y_test_rescaled, label='Real')
plt.plot(y_test_pred_rescaled, label='Predicción')
plt.title('Predicción temperatura diaria usando múltiples features')
plt.xlabel('Días')
plt.ylabel('Temperature (C)')
plt.legend()
plt.show()

In [ ]:
# ------------------------
# 7) Predecir los próximos 30 días
# ------------------------
model.eval()

# Tomamos los últimos 30 registros como semilla
# Para cada registro, usamos todas las columnas excepto 'Temperature (C)' como X
last_data = data.iloc[-30:].copy()

# Preprocesar igual que antes
# Categóricas
last_X_cat = ohe.transform(last_data[cat_cols])
# Numéricas
last_X_num = scaler.transform(last_data[num_cols])
# Concatenar
last_X_all = np.hstack([last_X_num, last_X_cat])

# Convertir a lista para actualizar con predicciones
last_X_list = last_X_all.tolist()
future_preds = []

for i in range(30):
    # Tomar último registro como input
    input_tensor = torch.tensor(last_X_list[-1], dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        pred_scaled = model(input_tensor).item()
    future_preds.append(pred_scaled)
    
    # Actualizar la fila siguiente
    # Para features que no dependen de la temperatura, podemos repetir la última fila
    next_row = last_X_list[-1].copy()
    # Si quieres, puedes ajustar otras columnas (ej. fecha) manualmente
    last_X_list.append(next_row)

# Revertir escala para ver temperaturas reales
future_preds_rescaled = y_scaler.inverse_transform(np.array(future_preds).reshape(-1,1))

# Crear fechas futuras
last_date = data['Formatted Date'].iloc[-1]
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=30, freq='D')

# DataFrame de predicciones
future_df = pd.DataFrame({'Predicted Temperature (C)': future_preds_rescaled.flatten()}, index=future_dates)

# Mostrar
print(future_df)

# ------------------------
# 8) Graficar predicciones futuras
# ------------------------
plt.figure(figsize=(12,5))
plt.plot(data['Formatted Date'][-60:], data['Temperature (C)'][-60:], label='Últimos 60 días reales')
plt.plot(future_df.index, future_df['Predicted Temperature (C)'], label='Predicción próximos 30 días', marker='o')
plt.title('Predicción de temperatura para los próximos 30 días')
plt.xlabel('Fecha')
plt.ylabel('Temperature (C)')
plt.legend()
plt.show()
